# 01 - Exploratory Data Analysis (EDA)
## Lending Club Loan Dataset
First look at the data: default rates, loan characteristics, and financial ratios

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import snowflake.connector
import os
from dotenv import load_dotenv

# Load credentials
load_dotenv()

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Libraries imported")

## Connect to Snowflake

In [ ]:
def get_connection():
    """Connect to Snowflake"""
    conn = snowflake.connector.connect(
        user=os.getenv('SNOWFLAKE_USER'),
        password=os.getenv('SNOWFLAKE_PASSWORD'),
        account=os.getenv('SNOWFLAKE_ACCOUNT'),
        warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
        database=os.getenv('SNOWFLAKE_DATABASE'),
        schema=os.getenv('SNOWFLAKE_SCHEMA')
    )
    return conn

conn = get_connection()
cursor = conn.cursor()
print("✅ Connected to Snowflake")

## Load Data from Snowflake

In [ ]:
# Load data from Snowflake
query = "SELECT * FROM LOANS LIMIT 100000"
df = pd.read_sql(query, conn)

print(f"✅ Loaded {len(df)} rows")
print(f"   Columns: {len(df.columns)}")
print(f"   Shape: {df.shape}")

## Dataset Overview

In [ ]:
# Display basic info
print("Dataset Info:")
print(f"  Rows: {df.shape[0]}")
print(f"  Columns: {df.shape[1]}")
print(f"  Date range: {df['ISSUE_D'].min()} to {df['ISSUE_D'].max()}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

# Display first few rows
df.head(3)

## Loan Status Distribution

In [ ]:
# Loan status breakdown
status_counts = df['LOAN_STATUS'].value_counts()
print("Loan Status Distribution:")
print(status_counts)
print(f"\nDefault Rate: {status_counts.get('Charged Off', 0) / len(df) * 100:.2f}%")

# Visualization
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
status_counts.plot(kind='bar', ax=ax[0], color='steelblue')
ax[0].set_title('Loan Status Distribution', fontsize=14, fontweight='bold')
ax[0].set_ylabel('Count')
ax[0].tick_params(axis='x', rotation=45)

# Pie chart
colors = ['#d62728' if x == 'Charged Off' else '#1f77b4' for x in status_counts.index]
status_counts.plot(kind='pie', ax=ax[1], autopct='%1.1f%%', colors=colors)
ax[1].set_title('Loan Status (%)' , fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/01_loan_status.png', dpi=300, bbox_inches='tight')
plt.show()

## Loan Grade Analysis

In [ ]:
# Default rate by grade
grade_default = df.groupby('GRADE').apply(
    lambda x: (x['LOAN_STATUS'] == 'Charged Off').sum() / len(x) * 100
)

print("Default Rate by Grade:")
print(grade_default.round(2))

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
grade_default.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Default Rate by Loan Grade', fontsize=14, fontweight='bold')
ax.set_ylabel('Default Rate (%)')
ax.set_xlabel('Loan Grade')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('outputs/02_default_by_grade.png', dpi=300, bbox_inches='tight')
plt.show()

## Key Financial Metrics

In [ ]:
# Summary statistics
print("Loan Amount Distribution:")
print(df['LOAN_AMOUNT'].describe())

print("\nInterest Rate Distribution:")
print(df['INT_RATE'].describe())

print("\nAnnual Income Distribution:")
print(df['ANNUAL_INC'].describe())

print("\nDTI (Debt-to-Income) Distribution:")
print(df['DTI'].describe())

## Missing Data Analysis

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing_Count': missing.values,
    'Missing_Percentage': missing_pct.values
}).sort_values('Missing_Percentage', ascending=False)

print("Missing Values (Top 20):")
print(missing_df.head(20).to_string(index=False))

## Feature Engineering Preview

## Conclusion
**Phase 1 Summary:**
- ✅ Data loaded from Snowflake (100k+ rows)
- ✅ Default distribution analyzed
- ✅ Loan grades and risk tiers identified
- ✅ Key financial metrics explored
- ✅ Feature engineering templates created

**Next Steps (Phase 2):**
- Deep dive into debt-to-income ratios
- Analyze FICO scores and credit metrics
- Build feature set for XGBoost model
- Create train/test split

In [ ]:
# Close connection
conn.close()
print("✅ Snowflake connection closed")